# 单端口分级校准

## 简介

单端口网络分析仪可用于测量双端口设备，前提是该设备是互易的。这是通过执行两次校准来完成的，这就是为什么它被称为*分级*校准。

首先，像正常情况一样在测试端口对 VNA 进行校准。这称为*第一层*。接下来，将设备连接到测试端口，并在设备的远端执行校准，即*第二层*。下图显示了一个框图，

![box diagram](oneport_tiered_calibration/images/boxDiagram.svg)

本笔记本将演示如何使用 [skrf](http://scikit-rf.org) 进行两级单端口校准。我们将使用用于表征波导到 CPW 探针的数据。因此，对于这个特定示例，上图的框图如下：

![probe diagram](oneport_tiered_calibration/images/probe.svg)

## 一些数据

可用的数据是文件夹 `'tier1/'` 和 `'tier2/'`。

In [ ]:
!ls {"oneport_tiered_calibration/"}

（如果您没有这些示例的 git 仓库，此笔记本的数据可以在[这里](https://github.com/scikit-rf/scikit-rf/tree/master/doc/source/examples/metrology/oneport_tiered_calibration)找到）

在每个文件夹中，您将找到两个子文件夹，称为 `'ideals/' ` 和 `'measured/'`。这些分别包含校准标准理想响应和测量响应的 touchstone 文件。

In [ ]:
!ls {"oneport_tiered_calibration/tier1/"}

第一层在波导接口处，由以下标准集组成

* short (短路)
* delay short (延迟短路)
* load (负载)
* radiating open (辐射开路，字面意义上是开路波导)

In [ ]:
!ls {"oneport_tiered_calibration/tier1/measured/"}

## 创建校准

### 第一层

首先定义 *第一层* 的校准

In [ ]:
import matplotlib.pyplot as plt

import skrf as rf
from skrf.calibration import OnePort
from skrf.io.general import read_all_networks

%matplotlib inline

rf.stylely()


tier1_ideals = read_all_networks('oneport_tiered_calibration/tier1/ideals/')
tier1_measured = read_all_networks('oneport_tiered_calibration/tier1/measured/')


tier1 = OnePort(measured = tier1_measured,
                ideals = tier1_ideals,
                name = 'tier1',
                sloppy_input=True)
tier1

因为我们保存了名称相同的对应 *理想* 和 *测量* 标准，校准将在初始化时自动对齐我们的标准。（有关创建校准对象的更多信息，可以在[文档](../../tutorials/Calibration.ipynb)中找到。）

对于第二层 2 同理，

### Tier 2

In [ ]:
tier2_ideals = read_all_networks('oneport_tiered_calibration/tier2/ideals/')
tier2_measured = read_all_networks('oneport_tiered_calibration/tier2/measured/')


tier2 = OnePort(measured = tier2_measured,
                ideals = tier2_ideals,
                name = 'tier2',
                sloppy_input=True)
tier2

## 误差网络

每个单端口校准包含一个双端口误差网络，该网络从计算的误差系数确定。*tier1* 的误差网络模拟 VNA，而 *tier2* 的误差网络表示 VNA **和** DUT。这些可以通过参数 `'error_ntwk'` 进行可视化。

对于第一层，

In [ ]:
tier1.error_ntwk.plot_s_db()
plt.title('Tier 1 Error Network')

对于第二层同理，

In [ ]:
tier2.error_ntwk.plot_s_db()
plt.title('Tier 2 Error Network')

## 消除 DUT 寄生效应

如前所述，*tier1* 的误差网络模拟 VNA，而 *tier2* 的误差网络表示 VNA+DUT。因此要确定 DUT 的响应，我们将 VNA 的反向 S 参数与 VNA+DUT 级联。

$$ DUT = VNA^{-1}\cdot (VNA \cdot DUT)$$

在 skrf 中，这是这样做的

In [ ]:
dut = tier1.error_ntwk.inv ** tier2.error_ntwk
dut.name = 'probe'
dut.plot_s_db()
plt.title('Probe S-parameters')
plt.ylim(-60,10)

您可能希望将其保存到磁盘以供将来使用，

    dut.write_touchstone()

In [ ]:
!ls {"probe*"}